In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.text_normalize import normalize_text

In [2]:
def check(raw, expected):
    got = normalize_text(raw)
    assert got == expected, f"expected {expected!r}, got {got!r}"

check("\u0643\u0627\u0644\u0627", "\u06A9\u0627\u0644\u0627")      # Arabic kaf -> Persian kaf
check("خوب   بود\n\nعالی", "خوب بود عالی")                           # spaces and line breaks
check("می\u200cخواهم", "می\u200cخواهم")                              # half-space must stay
check("a\u200bb\u200fc", "abc")                                      # invisible characters removed
check("قیمت ۱۲۰۰ و ٣٤ و 56", "قیمت 1200 و 34 و 56")                # digits unified
check("سایت https://x.com/a?b=1 خوب", "سایت خوب")                   # URL removed
check("عالی 😍", "عالی 😍")                                          # emoji kept
check("نه، اصلاً خوب نیست", "نه، اصلاً خوب نیست")                  # negation kept
check("  \t خوبي\u00a0و\r\nكيفيت ", "خوبی و کیفیت")                 # mixed case
check(None, "")
print("all checks passed")

all checks passed


In [3]:
import duckdb

con = duckdb.connect()
con.sql("""
    CREATE OR REPLACE TEMP VIEW comments AS
    SELECT * FROM read_csv('../data/digikala-comments.csv', all_varchar=true)
""")

sample = con.sql(r"""
    SELECT id, body
    FROM comments
    WHERE body LIKE '%ي%' OR body LIKE '%ك%'
       OR regexp_matches(body, '[\x{06F0}-\x{06F9}]')
    LIMIT 8
""").df()

sample["normalized"] = sample["body"].apply(normalize_text)
sample

,id,body,normalized
0,45354820,به نظرم چیز خوب و خفنیه مخصوصا با قیمت ۱۰ ت \r...,به نظرم چیز خوب و خفنیه مخصوصا با قیمت 10 ت من...
1,17911442,من چند رنگش رو سفارش دادم.خیلی خوش رنگ بودن تو...,من چند رنگش رو سفارش دادم.خیلی خوش رنگ بودن تو...
2,47241861,اینا رو بیرون دونه ای ۵ یا هفت میفروشن \r\nالا...,اینا رو بیرون دونه ای 5 یا هفت میفروشن الان 6 ...
3,25986578,خیلی خوشگل و کاربردی هستش و اندازه مناسبی داره...,خیلی خوشگل و کاربردی هستش و اندازه مناسبی داره...
4,29582488,سلام من سایه آکوآ رنگ ۷۴ سفارش دادم طبق نظرلت ...,سلام من سایه آکوآ رنگ 74 سفارش دادم طبق نظرلت ...
5,19945372,خیلی نسبت به قیمتش فقط دوستان بدونین این ماسک ...,خیلی نسبت به قیمتش فقط دوستان بدونین این ماسک ...
6,53894544,خوبه\r\nهمشون سالم بودن\r\nتعداد درست ارسال شد...,خوبه همشون سالم بودن تعداد درست ارسال شده بود ...
7,52130458,تعداد توپک ها زیاده و هر کدوم حجم خوبی دارن\nق...,تعداد توپک ها زیاده و هر کدوم حجم خوبی دارن قی...
